In [3]:
!pip install ultralytics albumentations --quiet

import os
import random
import shutil
import cv2
import albumentations as A
import yaml
from tqdm import tqdm
from ultralytics import YOLO

print("Ultralytics version loaded")

Ultralytics version loaded


In [4]:
src_folder = '/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive'
dst_folder = '/kaggle/working/dataset_yolov11'

train_ratio = 0.8
valid_ratio = 0.1
img_size = 640

In [5]:
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(dst_folder, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_folder, split, 'labels'), exist_ok=True)

In [6]:
with open(os.path.join(src_folder, 'classes_vie.txt'), 'r', encoding='utf-8') as f:
    classes = [line.strip() for line in f.readlines()]

print("Number of classes:", len(classes))

Number of classes: 52


In [7]:
all_images = [
    os.path.join(src_folder, 'images', f)
    for f in os.listdir(os.path.join(src_folder, 'images'))
    if f.endswith('.jpg')
]

random.shuffle(all_images)

train_size = int(len(all_images) * train_ratio)
valid_size = int(len(all_images) * valid_ratio)

train_images = all_images[:train_size]
valid_images = all_images[train_size:train_size+valid_size]
test_images = all_images[train_size+valid_size:]

print(len(train_images), len(valid_images), len(test_images))

2572 321 323


In [8]:
augmentations = A.Compose([
    A.Resize(img_size, img_size),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.Affine(shear=5, p=0.3),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.HueSaturationValue(p=0.3),
],
bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels']
))

In [9]:
def process_and_save(images, split):
    for img_path in tqdm(images, desc=f'Processing {split}'):
        filename = os.path.basename(img_path)
        label_filename = filename.replace('.jpg', '.txt')

        image = cv2.imread(img_path)
        if image is None:
            continue

        label_path = os.path.join(src_folder, 'labels', label_filename)
        if not os.path.exists(label_path):
            continue

        with open(label_path, 'r') as f:
            labels = [list(map(float, line.split())) for line in f.readlines()]

        if len(labels) == 0:
            continue

        bboxes = [label[1:] for label in labels]
        class_labels = [int(label[0]) for label in labels]

        # Validate bbox
        valid = True
        for bbox in bboxes:
            if not all(0 <= x <= 1 for x in bbox):
                valid = False
                break

        if not valid:
            continue

        try:
            if split == 'train':
                augmented = augmentations(
                    image=image,
                    bboxes=bboxes,
                    class_labels=class_labels
                )
                img_processed = augmented['image']
                bboxes_processed = augmented['bboxes']
                class_processed = augmented['class_labels']
            else:
                img_processed = cv2.resize(image, (img_size, img_size))
                bboxes_processed = bboxes
                class_processed = class_labels
        except:
            continue

        # Save image
        cv2.imwrite(os.path.join(dst_folder, split, 'images', filename), img_processed)

        # Save labels
        with open(os.path.join(dst_folder, split, 'labels', label_filename), 'w') as f:
            for bbox, cls_id in zip(bboxes_processed, class_processed):
                f.write(f"{cls_id} {' '.join(map(str, bbox))}\n")

In [10]:
process_and_save(train_images, 'train')
process_and_save(valid_images, 'valid')
process_and_save(test_images, 'test')

Processing test: 100%|██████████| 323/323 [00:06<00:00, 53.72it/s]


In [11]:
dataset_yaml = {
    'path': os.path.abspath(dst_folder),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': classes
}

with open(os.path.join(dst_folder, 'dataset.yaml'), 'w', encoding='utf-8') as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True)

print("dataset.yaml created")

dataset.yaml created


In [12]:
model = YOLO('yolo11n.pt')   # nano version

In [13]:
model.train(
    data=os.path.join(dst_folder, 'dataset.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,

    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    weight_decay=0.0005,

    close_mosaic=10,
    patience=20,
    workers=2,

    name="yolo11_vietnam_traffic_production"
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_yolov11/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_vietnam_traffic_production, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, ove

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 48, 49, 50, 51])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ca1236efef0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.

In [14]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,592,292 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2769.0±1045.0 MB/s, size: 155.8 KB)
val: Scanning /kaggle/working/dataset_yolov11/valid/labels.cache... 318 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 318/318 121.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 5.4it/s 3.7s0.1s
                   all        318        853      0.915      0.957      0.982      0.764
Đường người đi bộ cắt ngang         29         29      0.963          1      0.995      0.737
Đường giao nhau (ngã ba bên phải)          2          2          1      0.823      0.995      0.623
    Cấm đi ngược chiều         49         49      0.975      0.939       0.99      0.641
Phải đi vòng sang bên phải         49         49       0.96      0.918       0.99      0.627
Giao 

In [15]:
model.predict(
    source=os.path.join(dst_folder, 'test/images'),
    conf=0.25,
    save=True
)


image 1/319 /kaggle/working/dataset_yolov11/test/images/0002.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 2/319 /kaggle/working/dataset_yolov11/test/images/0029.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 3/319 /kaggle/working/dataset_yolov11/test/images/0033.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 4/319 /kaggle/working/dataset_yolov11/test/images/0039.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 5/319 /kaggle/working/dataset_yolov11/test/images/0055.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 6/319 /kaggle/working/dataset_yolov11/test/images/0057.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 7/319 /kaggle/working/dataset_yolov11/test/images/0065.jpg: 640x640 1 Cấm đỗ xe, 8.0ms
image 8/319 /kaggle/working/dataset_yolov11/test/images/0082.jpg: 640x640 1 Chỉ dành cho xe tải*, 2 Cấm rẽ trái và quay đầu xes, 8.0ms
image 9/319 /kaggle/working/dataset_yolov11/test/images/0096.jpg: 640x640 2 Cấm rẽ trái và quay đầu xes, 8.0ms
image 10/319 /kaggle/working/dataset_yolov11/test/images/0099.jpg: 640x640 2 Cấm rẽ trái và quay đầu x

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Đường người đi bộ cắt ngang', 1: 'Đường giao nhau (ngã ba bên phải)', 2: 'Cấm đi ngược chiều', 3: 'Phải đi vòng sang bên phải', 4: 'Giao nhau với đường đồng cấp', 5: 'Giao nhau với đường không ưu tiên', 6: 'Chỗ ngoặt nguy hiểm vòng bên trái', 7: 'Cấm rẽ trái', 8: 'Bến xe buýt', 9: 'Nơi giao nhau chạy theo vòng xuyến', 10: 'Cấm dừng và đỗ xe', 11: 'Chỗ quay xe', 12: 'Biển gộp làn đường theo phương tiện', 13: 'Đi chậm', 14: 'Cấm xe tải', 15: 'Đường bị thu hẹp về phía phải', 16: 'Giới hạn chiều cao', 17: 'Cấm quay đầu', 18: 'Cấm ô tô khách và ô tô tải', 19: 'Cấm rẽ phải và quay đầu', 20: 'Cấm ô tô', 21: 'Đường bị thu hẹp về phía trái', 22: 'Gồ giảm tốc phía trước', 23: 'Cấm xe hai và ba bánh', 24: 'Kiểm tra', 25: 'Chỉ dành cho xe máy*', 26: 'Chướng ngoại vật phía trước', 27: 'Trẻ em', 28: 'Xe tải và xe công*', 29: 'Cấm mô tô và xe máy', 3